# Scoring and transforming instances with biomolecular models

In [ ]:
import torch
from evedesign.system import System, Protein, SystemInstance, EntityInstance, Mutation
from evedesign.models.esm2 import ESM2
from evedesign.types import DeviceType

DEVICE: DeviceType = "cuda" if torch.cuda.is_available() else "cpu"

## Define system

For this example, we use E.coli beta-lactamase, with the signal peptide (1-23) removed from the target sequence.

In [ ]:
target_seq = "HPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCGAVLSRVDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHNMGDHVTRLDRWEPELNEAIPNDERDTTMPAAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWFIADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"

system = System(
    Protein(
        id="BLAT_ECOLX", rep=target_seq, first_index=24, sequences=None, structures=None
    )
)

We also set up a corresponding instance for the reference sequence. As we specified the full target sequence on the entity above, we could also use `system.rep_to_instance()` instead of explicitly defining the instance.

In [ ]:
target_instance = SystemInstance([
    EntityInstance(rep=target_seq)
])

## Set up model

Here, we use the ESM-2 LLM as an example model.

In [ ]:
esm = ESM2(
    model_name="esm2_t33_650M_UR50D"
).build(system)

## Functions for scoring

Note: which of the following functions are available depends on the particular model used

### General instance scoring with score()

This is a very general function for scoring a list of instances, and can be anything from a log-likelihood for a particular sequence to a predicted Tm score from a 3D structure prediction model. The score will be attached to the `score` attribute of the instance, a confidence value to the `confidence` attribute.

In [ ]:
scored_instances = esm.score([target_instance])

In [ ]:
scored_instances[0].score

### Mutation-based scoring

The following functions are only available for models that support the notion of a mutation relative to a defined target instance.

#### Single mutation scan

This function computes a single mutation matrix relative to a specified target instance (i.e., the self-substitution is assigned a score of 0). The matrix is computed for all positions across all entities by default, unless the `entity `and `positions` attributes are specified.

Note this function may use internal optimizations to speed up the calculation or give more accurate results compared to *score()*, e.g. by computing all substitutions for a position in one model forward pass, so should be used preferentially where possible.

In [ ]:
esm.single_mutation_scan(target_instance)

#### Arbitrary mutant scoring

This function allows to score arbitrary single and higher-order mutants relative to the target instance. Individual mutants are specified as a list of *Mutation* objects.

Note this function may use internal optimizations to speed up the calculation or give more accurate results compared to *score()*, so should be used preferentially where possible.

In [ ]:
# single mutant
single_mutant = [Mutation(entity=0, pos=180, ref="M", to="T")]
double_mutant = [Mutation(entity=0, pos=180, ref="M", to="T"), Mutation(entity=0, pos=236, ref="G", to="S"),]

scored_mutant_instances = esm.score_mutants(
    target_instance,
    [
        single_mutant,
        double_mutant,
    ]
)

[inst.score for inst in scored_mutant_instances]

#### Conditional mutation scoring with score_conditional()

This function is typically only of relevance for applications where the conditional likelihood P(x_i | x_\i) needs to be computed, e.g. for Gibbs sampling.

## Transform

This operation transforms instances from one representation level to another. In the case of ESM-2 shown here, it maps sequences to embeddings.

The *transform()* method may also set the *score* attribute of the instance if a score can be computed in the same model pass to make computations more efficient.

In [ ]:
instances_transformed = esm.transform([target_instance])

In [ ]:
# indices: first instance, first entity
instances_transformed[0][0].embedding

In [ ]:
instances_transformed[0].score